# Hai video Thành → ảnh cho Label Studio (10 ảnh/giây)

Upload hai video gốc thành một Kaggle Dataset riêng, Add Input vào notebook, rồi chạy các cell theo thứ tự. Notebook chỉ đọc input và tạo ảnh + manifest + XML + ZIP; không upload sang dịch vụ khác, không train, không tự gán nhãn AI.

Sau khi chạy, tải ZIP trong Output về, giải nén rồi upload các JPG vào project pose13 trong Label Studio local. Giữ frames_manifest.json đi kèm để chuyển nhãn/tra ngược về video. XML được đóng gói để tạo project ảnh mới nếu cần; backend L chạy trên máy của bạn.

Tên ảnh chứa video hash và số frame gốc. Manifest giữ PTS nguồn; timestamp_relative_seconds bắt đầu tại frame đầu của video, source_timestamp_seconds giữ PTS gốc. Thời gian lấy mẫu mục tiêu khác thời gian thực của frame được chọn. Notebook không dùng JSON nhãn video cũ và không tự tạo nhãn cho frame.

**Đã chốt:** cả hai video, đều10ảnh/giây; giữ kích thước/hướng hiển thị gốc. Nếu sửa cấu hình sau này, dùng thư mục output mới để không trộn các lượt.

Kaggle đã đổi tên hai file tải lên thành `Thnh.MOV` và `Thnh (1).MOV`. Notebook tìm đúng hai tên này; manifest dùng SHA-256 video để đối chiếu với video gốc.


In [ ]:
from pathlib import Path
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/thanh_pose13_10fps")
SAMPLE_FPS = 10
# Để [] để tìm hai tên mà Kaggle đã lưu sau khi bỏ dấu tiếng Việt. Nếu Kaggle đổi tên file, điền HAI đường dẫn thực tế.
VIDEO_PATHS = []
EXPECTED_NAMES = ["Thnh.MOV", "Thnh (1).MOV"]


In [ ]:
import bisect, hashlib, json, math, shutil, subprocess, unicodedata
from fractions import Fraction
from PIL import Image

def run_checked(args):
    result = subprocess.run([str(x) for x in args], capture_output=True, text=True, encoding="utf-8", errors="replace")
    if result.returncode:
        raise RuntimeError(result.stderr[-4000:])
    return result.stdout

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def discover_videos(input_root, explicit, names):
    if explicit:
        paths = [Path(p) for p in explicit]
        if len(paths) != 2 or len({p.resolve() for p in paths}) != 2:
            raise ValueError("Cần đúng hai đường dẫn video khác nhau.")
        if not all(p.is_file() for p in paths):
            raise FileNotFoundError("Một video đã chỉ định không tồn tại.")
        return paths
    candidates = [p for p in input_root.rglob("*") if p.is_file() and p.suffix.lower() in {".mov", ".mp4"}]
    norm = lambda s: unicodedata.normalize("NFC", s).casefold()
    paths = []
    for name in names:
        matches = [p for p in candidates if norm(p.name) == norm(name)]
        if len(matches) != 1:
            inventory = "\n".join(str(p) for p in candidates)
            raise ValueError(f"Không tìm được duy nhất {name}. Điền VIDEO_PATHS từ danh sách:\n{inventory}")
        paths.append(matches[0])
    return paths

def probe_video(path):
    raw = json.loads(run_checked(["ffprobe", "-v", "error", "-select_streams", "v:0",
        "-show_frames", "-show_streams", "-show_entries",
        "stream=width,height,time_base:stream_side_data=rotation:frame=best_effort_timestamp",
        "-of", "json", path]))
    streams = raw.get("streams", [])
    if len(streams) != 1:
        raise ValueError("Cần một video stream được chọn.")
    stream = streams[0]
    ticks = [int(f["best_effort_timestamp"]) for f in raw.get("frames", [])]
    if not ticks:
        raise ValueError("Không có frame hoặc thiếu timestamp.")
    tb = Fraction(stream["time_base"])
    times = [float(t * tb) for t in ticks]
    if any(b <= a for a, b in zip(times, times[1:])):
        raise ValueError("PTS trùng/không tăng: cần kiểm tra nguồn trước khi tự ghép ảnh.")
    return stream, ticks, times

def choose_frames(times, fps):
    if not times or not math.isfinite(fps) or not 0 < fps <= 30:
        raise ValueError("Cần timestamp và tần suất hợp lệ (0,30].")
    if any(not math.isfinite(t) for t in times) or any(b <= a for a,b in zip(times,times[1:])):
        raise ValueError("Timestamp phải hữu hạn và tăng nghiêm ngặt.")
    relative = [t-times[0] for t in times]
    rows, used = [], set()
    count = math.floor(relative[-1] * fps + 1e-8) + 1
    for k in range(count):
        target = k / fps
        at = bisect.bisect_left(relative, target)
        options = [i for i in (at-1, at) if 0 <= i < len(relative)]
        index = min(options, key=lambda i: (abs(relative[i]-target), i))
        if index not in used:
            used.add(index)
            rows.append((index, target))
    return rows

def extract_one(path, output, fps):
    stream, ticks, times = probe_video(path)
    chosen = choose_frames(times, fps)
    digest = sha256_file(path)
    prefix = f"video_{digest[:16]}"
    images = output / "images"
    images.mkdir(parents=True, exist_ok=True)
    # FFmpeg select nhận index frame giải mã; không đổi FPS hay nội suy hình.
    expression = "+".join(f"eq(n\\,{index})" for index, _ in chosen)
    pattern = images / f"{prefix}_sample_%06d.jpg"
    run_checked(["ffmpeg", "-hide_banner", "-loglevel", "error", "-nostdin", "-n",
        "-i", path, "-map", "0:v:0", "-vf", f"select={expression}",
        "-fps_mode", "vfr", "-q:v", "2", "-start_number", "0", pattern])
    samples = sorted(images.glob(f"{prefix}_sample_*.jpg"))
    if len(samples) != len(chosen):
        raise ValueError(f"Số ảnh {len(samples)} không khớp số frame chọn {len(chosen)}.")
    rows = []
    for order, (index, target_time) in enumerate(chosen):
        old = images / f"{prefix}_sample_{order:06d}.jpg"
        new = images / f"{prefix}_frame_{index:06d}.jpg"
        if new.exists():
            raise FileExistsError(new)
        old.rename(new)
        with Image.open(new) as im:
            width, height = im.size
            im.verify()
        rows.append({
            "image_id": new.stem, "filename": new.name,
            "image_sha256": sha256_file(new), "source_video": path.name,
            "source_video_sha256": digest, "source_frame_index": index,
            "source_pts": ticks[index], "time_base": stream["time_base"],
            "source_timestamp_seconds": times[index],
            "timestamp_relative_seconds": times[index]-times[0],
            "sampling_target_seconds": target_time, "sampling_fps": fps,
            "width": width, "height": height,
            "source_rotation_metadata": stream.get("side_data_list", [])
        })
    return rows

# Kiểm tra hợp đồng sampling trên timeline CFR và VFR, không đọc video người dùng.
assert [i for i,t in choose_frames([i/30 for i in range(30)],10)] == list(range(0,30,3))
assert choose_frames([2.0],10) == [(0,0.0)]
assert choose_frames([1.0,1.03,1.11,1.19],10) == [(0,0.0),(2,0.1)]
assert len({i for i,t in choose_frames([0,0.5,1.0],10)}) == 3
for bad in ([], [0,0], [0,float("nan")]):
    try:
        choose_frames(bad,10)
    except ValueError:
        pass
    else:
        raise AssertionError("Phải từ chối timestamp lỗi.")
print("Sampling contract tests passed.")


In [ ]:
LABEL_CONFIG = "<View><Header value=\"Pose 13 điểm: một đầu + 12 khớp cơ thể\" /><Header value=\"Ẩn điểm dưới confidence 5%; điểm chồng nhau vẫn giữ. Nhãn AI nháp, cần sửa trước khi Submit.\" /><Image name=\"image\" value=\"$image\" zoom=\"true\" /><RectangleLabels name=\"person\" toName=\"image\"><Label value=\"person\" background=\"#3b82f6\" /></RectangleLabels><KeyPointLabels name=\"keypoints\" toName=\"image\"><Label value=\"head\" model_index=\"0\" background=\"#ef4444\" /><Label value=\"left_shoulder\" model_index=\"1\" background=\"#22c55e\" /><Label value=\"right_shoulder\" model_index=\"2\" background=\"#f59e0b\" /><Label value=\"left_elbow\" model_index=\"3\" background=\"#22c55e\" /><Label value=\"right_elbow\" model_index=\"4\" background=\"#f59e0b\" /><Label value=\"left_wrist\" model_index=\"5\" background=\"#22c55e\" /><Label value=\"right_wrist\" model_index=\"6\" background=\"#f59e0b\" /><Label value=\"left_hip\" model_index=\"7\" background=\"#22c55e\" /><Label value=\"right_hip\" model_index=\"8\" background=\"#f59e0b\" /><Label value=\"left_knee\" model_index=\"9\" background=\"#22c55e\" /><Label value=\"right_knee\" model_index=\"10\" background=\"#f59e0b\" /><Label value=\"left_ankle\" model_index=\"11\" background=\"#22c55e\" /><Label value=\"right_ankle\" model_index=\"12\" background=\"#f59e0b\" /></KeyPointLabels></View>"
# Cần ffmpeg/ffprobe có sẵn trong phiên Kaggle. Không tự cài/upgrade package.
for tool in ("ffmpeg", "ffprobe"):
    if not shutil.which(tool):
        raise RuntimeError(f"Thiếu {tool}; cài FFmpeg trong phiên rồi chạy lại.")
videos = discover_videos(INPUT_ROOT, VIDEO_PATHS, EXPECTED_NAMES)
if OUTPUT_ROOT.exists():
    raise FileExistsError("Output đã tồn tại. Chọn OUTPUT_ROOT mới để không ghi đè/trộn ảnh.")
OUTPUT_ROOT.mkdir(parents=True)
manifest = []
for video in videos:
    print("Đang tách:", video.name)
    rows = extract_one(video, OUTPUT_ROOT, SAMPLE_FPS)
    manifest.extend(rows)
    print("Đã kiểm tra:", len(rows), "ảnh")
if len({r["image_id"] for r in manifest}) != len(manifest):
    raise ValueError("Hai video có cùng nội dung/mã ảnh; cần kiểm tra trước khi import.")
(OUTPUT_ROOT / "frames_manifest.json").write_text(json.dumps({
    "schema_version": 1, "source_frame_index_base": 0,
    "sample_fps": SAMPLE_FPS, "frames": manifest
}, ensure_ascii=False, indent=2), encoding="utf-8")
(OUTPUT_ROOT / "pose13-labeling.xml").write_text(LABEL_CONFIG, encoding="utf-8")
(OUTPUT_ROOT / "IMPORT.txt").write_text(
    "Giải nén ZIP. Upload các JPG trong images vào project ảnh pose13 của Label Studio local.\n"
    "Không import manifest như task: đây là bảng nguồn đi kèm.\n"
    "Giữ manifest nguyên vẹn để ghép nhãn bằng tên ảnh; không dựa thứ tự task.\n"
    "Kết nối backend local9090. Prediction là nháp; sửa bbox/pose trước Submit.\n"
    "Notebook chưa tạo nhãn hoặc train. JSON13 cần converter riêng nếu xuất YOLO TXT.\n",
    encoding="utf-8")
archive = shutil.make_archive(str(OUTPUT_ROOT), "zip", root_dir=OUTPUT_ROOT)
print("Tổng ảnh đã xác minh:", len(manifest))
print("Tải file:", archive)
from IPython.display import FileLink, display
display(FileLink(archive))
